In [1]:
import polars as pl
from pathlib import Path

In [3]:
FILE = Path('../../data/window_functions_salles.pq')
df = pl.read_parquet(FILE)
print(df)

shape: (24, 5)
┌────────┬─────────┬─────────┬────────┬──────────┐
│ regiao ┆ produto ┆ mes     ┆ vendas ┆ renda    │
│ ---    ┆ ---     ┆ ---     ┆ ---    ┆ ---      │
│ str    ┆ str     ┆ str     ┆ i64    ┆ f64      │
╞════════╪═════════╪═════════╪════════╪══════════╡
│ norte  ┆ A       ┆ 2026-01 ┆ 152    ┆ 166867.0 │
│ norte  ┆ A       ┆ 2026-02 ┆ 142    ┆ 123694.0 │
│ norte  ┆ A       ┆ 2026-03 ┆ 156    ┆ 139879.0 │
│ norte  ┆ A       ┆ 2026-04 ┆ 238    ┆ 74886.0  │
│ norte  ┆ A       ┆ 2026-05 ┆ 171    ┆ 188266.0 │
│ …      ┆ …       ┆ …       ┆ …      ┆ …        │
│ sul    ┆ B       ┆ 2026-02 ┆ 239    ┆ 139346.0 │
│ sul    ┆ B       ┆ 2026-03 ┆ 157    ┆ 76886.0  │
│ sul    ┆ B       ┆ 2026-04 ┆ 293    ┆ 51551.0  │
│ sul    ┆ B       ┆ 2026-05 ┆ 298    ┆ 31394.0  │
│ sul    ┆ B       ┆ 2026-06 ┆ 278    ┆ 23890.0  │
└────────┴─────────┴─────────┴────────┴──────────┘


## Ranking

| method | SQL |
|---|---|
| *ordinal* | `ROW_NUMBER()` |
| *min* | `RANK()` |
| *dense* | `DENSE_RANK()` |
| *average* | Média das posições em empate (sem equivalente no SQL) |
| *max* | Usa a maior posição entre empatados |

In [18]:
df.with_columns(
    pl.col('mes')
    .rank(method='dense', descending=False)
    .over('regiao')
    .alias('primeiro_dia_de_vendas_por_regiao')
)

regiao,produto,mes,vendas,renda,primeiro_dia_de_vendas_por_regiao
str,str,str,i64,f64,u32
"""norte""","""A""","""2026-01""",152,166867.0,1
"""norte""","""A""","""2026-02""",142,123694.0,2
"""norte""","""A""","""2026-03""",156,139879.0,3
"""norte""","""A""","""2026-04""",238,74886.0,4
"""norte""","""A""","""2026-05""",171,188266.0,5
…,…,…,…,…,…
"""sul""","""B""","""2026-02""",239,139346.0,2
"""sul""","""B""","""2026-03""",157,76886.0,3
"""sul""","""B""","""2026-04""",293,51551.0,4


## Agregações
Diferentemente do comando `group_by().agg()` que reduz as linhas, ao usar `over()` mantém todas as linhas originais e repete o valor agregado em cada registro, exatamente como uma `window_function`. Isso é exatamente o que o `SQL` faz.

In [27]:
print(df.with_columns(
    pl.col('vendas').mean().over('regiao').alias('media_vendas_regiao'),
    pl.col('vendas').sum().over('regiao').alias('soma_vendas_regiao'),
    pl.col('vendas').max().over('regiao').alias('max_vendas_regiao'),
    pl.col('vendas').min().over('regiao').alias('min_vendas_regiao'),
    pl.col('vendas').count().over('regiao').alias('count_vendas_regiao'),
    pl.col('vendas').std().over('regiao').alias('std_vendas_regiao'),
).select([
    'regiao',
    'vendas',
    'media_vendas_regiao',
    'soma_vendas_regiao',
    'max_vendas_regiao',
    'min_vendas_regiao',
    'count_vendas_regiao',
    'std_vendas_regiao',
]))

shape: (24, 8)
┌────────┬────────┬────────────────┬────────────────┬────────────────┬────────────────┬────────────────┬───────────────┐
│ regiao ┆ vendas ┆ media_vendas_r ┆ soma_vendas_re ┆ max_vendas_reg ┆ min_vendas_reg ┆ count_vendas_r ┆ std_vendas_re │
│ ---    ┆ ---    ┆ egiao          ┆ giao           ┆ iao            ┆ iao            ┆ egiao          ┆ giao          │
│ str    ┆ i64    ┆ ---            ┆ ---            ┆ ---            ┆ ---            ┆ ---            ┆ ---           │
│        ┆        ┆ f64            ┆ i64            ┆ i64            ┆ i64            ┆ u32            ┆ f64           │
╞════════╪════════╪════════════════╪════════════════╪════════════════╪════════════════╪════════════════╪═══════════════╡
│ norte  ┆ 152    ┆ 161.333333     ┆ 1936           ┆ 252            ┆ 87             ┆ 12             ┆ 47.442469     │
│ norte  ┆ 142    ┆ 161.333333     ┆ 1936           ┆ 252            ┆ 87             ┆ 12             ┆ 47.442469     │
│ norte  ┆ 156   

## LAG e LEAD

In [35]:
(
    df
    .group_by('mes').agg(
        sum_salles=pl.col('vendas').sum()
    ).sort('mes')
    .with_columns(
        pl.col('sum_salles').shift(1).alias('past_salles'),
        pl.col('sum_salles').shift(-1).alias('next_salles')
    )
)

mes,sum_salles,past_salles,next_salles
str,i64,i64,i64
"""2026-01""",794,null,605
"""2026-02""",605,794,761
"""2026-03""",761,605,852
"""2026-04""",852,761,875
"""2026-05""",875,852,874
"""2026-06""",874,875,null


## Cum Sum

In [38]:
df.with_columns(
    pl.col("vendas").cum_sum().over("regiao").alias("acumulado"),
    pl.col("vendas").cum_max().over("regiao").alias("max_acumulado"),
    pl.col("vendas").cum_min().over("regiao").alias("min_acumulado"),
)

regiao,produto,mes,vendas,renda,acumulado,max_acumulado,min_acumulado
str,str,str,i64,f64,i64,i64,i64
"""norte""","""A""","""2026-01""",152,166867.0,152,152,152
"""norte""","""A""","""2026-02""",142,123694.0,294,152,142
"""norte""","""A""","""2026-03""",156,139879.0,450,156,142
"""norte""","""A""","""2026-04""",238,74886.0,688,238,142
"""norte""","""A""","""2026-05""",171,188266.0,859,238,142
…,…,…,…,…,…,…,…
"""sul""","""B""","""2026-02""",239,139346.0,1799,269,71
"""sul""","""B""","""2026-03""",157,76886.0,1956,269,71
"""sul""","""B""","""2026-04""",293,51551.0,2249,293,71


## Médias Moveis

In [39]:
df.with_columns(
    pl.col("vendas").rolling_mean(window_size=3).over("regiao").alias("media_movel_3"),
    pl.col("vendas").rolling_sum(window_size=3).over("regiao").alias("soma_movel_3"),
    pl.col("vendas").rolling_max(window_size=3).over("regiao").alias("max_movel_3"),
    pl.col("vendas").rolling_min(window_size=3).over("regiao").alias("min_movel_3"),
    pl.col("vendas").rolling_std(window_size=3).over("regiao").alias("std_movel_3"),
)

regiao,produto,mes,vendas,renda,media_movel_3,soma_movel_3,max_movel_3,min_movel_3,std_movel_3
str,str,str,i64,f64,f64,i64,i64,i64,f64
"""norte""","""A""","""2026-01""",152,166867.0,null,null,null,null,null
"""norte""","""A""","""2026-02""",142,123694.0,null,null,null,null,null
"""norte""","""A""","""2026-03""",156,139879.0,150.0,450,156,142,7.211103
"""norte""","""A""","""2026-04""",238,74886.0,178.666667,536,238,142,51.858783
"""norte""","""A""","""2026-05""",171,188266.0,188.333333,565,238,156,43.661577
…,…,…,…,…,…,…,…,…,…
"""sul""","""B""","""2026-02""",239,139346.0,245.0,735,257,239,10.392305
"""sul""","""B""","""2026-03""",157,76886.0,211.666667,635,239,157,47.342722
"""sul""","""B""","""2026-04""",293,51551.0,229.666667,689,293,157,68.478707


## NTILE
Faz uma divisão de *n* partes nos seus dados, por exemplo se você possui 100 linhas e divide em 4 partes teremos 25 linhas em cada parte dado a coluna que você utilizou para divisão.

In [41]:
df.with_columns(
    pl.col("renda").qcut(4, labels=["1", "2", "3", "4"]).over("regiao").alias("faixa_renda")
)

regiao,produto,mes,vendas,renda,faixa_renda
str,str,str,i64,f64,cat
"""norte""","""A""","""2026-01""",152,166867.0,"""4"""
"""norte""","""A""","""2026-02""",142,123694.0,"""3"""
"""norte""","""A""","""2026-03""",156,139879.0,"""3"""
"""norte""","""A""","""2026-04""",238,74886.0,"""2"""
"""norte""","""A""","""2026-05""",171,188266.0,"""4"""
…,…,…,…,…,…
"""sul""","""B""","""2026-02""",239,139346.0,"""3"""
"""sul""","""B""","""2026-03""",157,76886.0,"""2"""
"""sul""","""B""","""2026-04""",293,51551.0,"""2"""
